# Import
 - We will import all the necessary libraries and tools

In [1]:
import tensorflow as tf
import pickle

from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# Data Generators ***with augmentation***

In [2]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True
)

val_test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    'dataset/train',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

validation_generator = val_test_datagen.flow_from_directory(
    'dataset/validation',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

test_generator = val_test_datagen.flow_from_directory(
    'dataset/test',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

Found 1145 images belonging to 6 classes.
Found 267 images belonging to 6 classes.
Found 220 images belonging to 6 classes.


# Save Class Mapping

In [3]:
with open("googlenet_googlenet_class_indices.pkl", "wb") as f:
    pickle.dump(train_generator.class_indices, f)

print("Classes:", train_generator.class_indices)

Classes: {'A1-Walking': 0, 'A2-Sitting-down': 1, 'A3-StandUp': 2, 'A4-PickObject': 3, 'A5-DrinkWater': 4, 'A6-Fall': 5}


# Load Base Model

In [4]:
base_model = InceptionV3(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze Layers ***Phase 1***

In [5]:
for layer in base_model.layers:
    layer.trainable = False

# Build Classifier Head

In [6]:
x = base_model.output
x = GlobalAveragePooling2D()(x)

x = BatchNormalization()(x)
x = Dropout(0.5)(x)

x = Dense(128, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.5)(x)

output = Dense(train_generator.num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=output)

# Compile Model

In [7]:
model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 111, 111,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 111, 111,  │         96 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 111, 111,  │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 109, 109,  │      9,216 │ activation[0][0]  │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 109, 109,  │         96 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 109, 109,  │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 109, 109,  │     18,432 │ activation_1[0][… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 109, 109,  │        192 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 109, 109,  │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 54, 54,    │          0 │ activation_2[0][… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 54, 54,    │      5,120 │ max_pooling2d[0]… │
│                     │ 80)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 54, 54,    │        240 │ conv2d_3[0][0]    │
│ (BatchNormalizatio… │ 80)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_3        │ (None, 54, 54,    │          0 │ batch_normalizat… │
│ (Activation)        │ 80)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 52, 52,    │    138,240 │ activation_3[0][… │
│                     │ 192)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 52, 52,    │        576 │ conv2d_4[0][0]    │
│ (BatchNormalizatio… │ 192)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_4        │ (None, 52, 52,    │          0 │ batch_normalizat

 Total params: 22,074,534 (84.21 MB)

 Trainable params: 267,398 (1.02 MB)

 Non-trainable params: 21,807,136 (83.19 MB)

# Callbacks

In [8]:
checkpoint = ModelCheckpoint(
    'googlenet_googlenet_model.h5',
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

early_stopping = EarlyStopping(
    monitor='val_accuracy',
    patience=8,
    restore_best_weights=True,
    verbose=1
)

# Train ***Phase 1***

In [9]:
history = model.fit(
    train_generator,
    epochs=50,
    validation_data=validation_generator,
    callbacks=[checkpoint, early_stopping]
)

c:\Users\harsh\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 714ms/step - accuracy: 0.1916 - loss: 2.8455
Epoch 1: val_accuracy improved from -inf to 0.31835, saving model to googlenet_googlenet_model.h5


36/36 ━━━━━━━━━━━━━━━━━━━━ 41s 960ms/step - accuracy: 0.1922 - loss: 2.8403 - val_accuracy: 0.3184 - val_loss: 1.7122
Epoch 2/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 751ms/step - accuracy: 0.2999 - loss: 2.3507
Epoch 2: val_accuracy improved from 0.31835 to 0.32959, saving model to googlenet_googlenet_model.h5


36/36 ━━━━━━━━━━━━━━━━━━━━ 33s 925ms/step - accuracy: 0.2997 - loss: 2.3499 - val_accuracy: 0.3296 - val_loss: 1.5862
Epoch 3/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 737ms/step - accuracy: 0.3631 - loss: 2.0227
Epoch 3: val_accuracy improved from 0.32959 to 0.33333, saving model to googlenet_googlenet_model.h5


36/36 ━━━━━━━━━━━━━━━━━━━━ 33s 910ms/step - accuracy: 0.3630 - loss: 2.0227 - val_accuracy: 0.3333 - val_loss: 1.4845
Epoch 4/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 732ms/step - accuracy: 0.3977 - loss: 1.9576
Epoch 4: val_accuracy improved from 0.33333 to 0.33708, saving model to googlenet_googlenet_model.h5


...

36/36 ━━━━━━━━━━━━━━━━━━━━ 35s 976ms/step - accuracy: 0.5587 - loss: 1.2733 - val_accuracy: 0.6854 - val_loss: 0.7798
Epoch 22/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 786ms/step - accuracy: 0.5678 - loss: 1.2358
Epoch 22: val_accuracy did not improve from 0.68539
36/36 ━━━━━━━━━━━━━━━━━━━━ 34s 951ms/step - accuracy: 0.5678 - loss: 1.2354 - val_accuracy: 0.6742 - val_loss: 0.7850
Epoch 23/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 808ms/step - accuracy: 0.5779 - loss: 1.1362
Epoch 23: val_accuracy improved from 0.68539 to 0.68914, saving model to googlenet_googlenet_model.h5


36/36 ━━━━━━━━━━━━━━━━━━━━ 37s 1s/step - accuracy: 0.5776 - loss: 1.1377 - val_accuracy: 0.6891 - val_loss: 0.7445
Epoch 24/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 814ms/step - accuracy: 0.5408 - loss: 1.2720
Epoch 24: val_accuracy improved from 0.68914 to 0.70037, saving model to googlenet_googlenet_model.h5


36/36 ━━━━━━━━━━━━━━━━━━━━ 36s 1s/step - accuracy: 0.5413 - loss: 1.2712 - val_accuracy: 0.7004 - val_loss: 0.7472
Epoch 25/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 813ms/step - accuracy: 0.5766 - loss: 1.2362
Epoch 25: val_accuracy improved from 0.70037 to 0.71536, saving model to googlenet_googlenet_model.h5


36/36 ━━━━━━━━━━━━━━━━━━━━ 36s 1s/step - accuracy: 0.5766 - loss: 1.2362 - val_accuracy: 0.7154 - val_loss: 0.7006
Epoch 26/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 806ms/step - accuracy: 0.5629 - loss: 1.2388
Epoch 26: val_accuracy improved from 0.71536 to 0.71910, saving model to googlenet_googlenet_model.h5


36/36 ━━━━━━━━━━━━━━━━━━━━ 38s 1s/step - accuracy: 0.5631 - loss: 1.2385 - val_accuracy: 0.7191 - val_loss: 0.7011
Epoch 27/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 837ms/step - accuracy: 0.5729 - loss: 1.2457
Epoch 27: val_accuracy did not improve from 0.71910
36/36 ━━━━━━━━━━━━━━━━━━━━ 37s 1s/step - accuracy: 0.5732 - loss: 1.2446 - val_accuracy: 0.7079 - val_loss: 0.7185
Epoch 28/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 897ms/step - accuracy: 0.5498 - loss: 1.2355
Epoch 28: val_accuracy did not improve from 0.71910
36/36 ━━━━━━━━━━━━━━━━━━━━ 39s 1s/step - accuracy: 0.5501 - loss: 1.2346 - val_accuracy: 0.7116 - val_loss: 0.7190
Epoch 29/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 866ms/step - accuracy: 0.5461 - loss: 1.2244
Epoch 29: val_accuracy did not improve from 0.71910
36/36 ━━━━━━━━━━━━━━━━━━━━ 38s 1s/step - accuracy: 0.5466 - loss: 1.2241 - val_accuracy: 0.6966 - val_loss: 0.7306
Epoch 30/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 874ms/step - accuracy: 0.5724 - loss: 1.1671
Epoch 30: val_accuracy did not impro

# Fine-Tuning ***Phase 2***

In [10]:
# Unfreeze top layers
for layer in base_model.layers[-50:]:
    layer.trainable = True

model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_finetune = model.fit(
    train_generator,
    epochs=30,
    validation_data=validation_generator,
    callbacks=[checkpoint, early_stopping]
)

Epoch 1/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 892ms/step - accuracy: 0.4571 - loss: 1.5716
Epoch 1: val_accuracy did not improve from 0.71910
36/36 ━━━━━━━━━━━━━━━━━━━━ 50s 1s/step - accuracy: 0.4567 - loss: 1.5723 - val_accuracy: 0.6891 - val_loss: 0.7662
Epoch 2/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 912ms/step - accuracy: 0.4479 - loss: 1.5664
Epoch 2: val_accuracy did not improve from 0.71910
36/36 ━━━━━━━━━━━━━━━━━━━━ 39s 1s/step - accuracy: 0.4481 - loss: 1.5652 - val_accuracy: 0.6704 - val_loss: 0.8136
Epoch 3/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 921ms/step - accuracy: 0.4922 - loss: 1.4591
Epoch 3: val_accuracy did not improve from 0.71910
36/36 ━━━━━━━━━━━━━━━━━━━━ 39s 1s/step - accuracy: 0.4927 - loss: 1.4586 - val_accuracy: 0.6517 - val_loss: 0.8315
Epoch 4/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 923ms/step - accuracy: 0.5253 - loss: 1.3309
Epoch 4: val_accuracy did not improve from 0.71910
36/36 ━━━━━━━━━━━━━━━━━━━━ 39s 1s/step - accuracy: 0.5255 - loss: 1.3308 - val_accuracy: 0.6592 - val_loss:

36/36 ━━━━━━━━━━━━━━━━━━━━ 40s 1s/step - accuracy: 0.5804 - loss: 1.1858 - val_accuracy: 0.7228 - val_loss: 0.7437
Epoch 11/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 915ms/step - accuracy: 0.5775 - loss: 1.2379
Epoch 11: val_accuracy improved from 0.72285 to 0.72659, saving model to googlenet_googlenet_model.h5


36/36 ━━━━━━━━━━━━━━━━━━━━ 40s 1s/step - accuracy: 0.5774 - loss: 1.2374 - val_accuracy: 0.7266 - val_loss: 0.7488
Epoch 12/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 853ms/step - accuracy: 0.5926 - loss: 1.1645
Epoch 12: val_accuracy did not improve from 0.72659
36/36 ━━━━━━━━━━━━━━━━━━━━ 37s 1s/step - accuracy: 0.5925 - loss: 1.1648 - val_accuracy: 0.7116 - val_loss: 0.7218
Epoch 13/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 908ms/step - accuracy: 0.5588 - loss: 1.2271
Epoch 13: val_accuracy did not improve from 0.72659
36/36 ━━━━━━━━━━━━━━━━━━━━ 39s 1s/step - accuracy: 0.5589 - loss: 1.2266 - val_accuracy: 0.7154 - val_loss: 0.7063
Epoch 14/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 921ms/step - accuracy: 0.5992 - loss: 1.1437
Epoch 14: val_accuracy did not improve from 0.72659
36/36 ━━━━━━━━━━━━━━━━━━━━ 41s 1s/step - accuracy: 0.5993 - loss: 1.1436 - val_accuracy: 0.7266 - val_loss: 0.7053
Epoch 15/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 970ms/step - accuracy: 0.6090 - loss: 1.0858
Epoch 15: val_accuracy did not impro

36/36 ━━━━━━━━━━━━━━━━━━━━ 40s 1s/step - accuracy: 0.6026 - loss: 1.1481 - val_accuracy: 0.7341 - val_loss: 0.6977
Epoch 19/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 903ms/step - accuracy: 0.5954 - loss: 1.1469
Epoch 19: val_accuracy did not improve from 0.73408
36/36 ━━━━━━━━━━━━━━━━━━━━ 39s 1s/step - accuracy: 0.5961 - loss: 1.1455 - val_accuracy: 0.7341 - val_loss: 0.6759
Epoch 20/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 911ms/step - accuracy: 0.5975 - loss: 1.1241
Epoch 20: val_accuracy did not improve from 0.73408
36/36 ━━━━━━━━━━━━━━━━━━━━ 39s 1s/step - accuracy: 0.5974 - loss: 1.1242 - val_accuracy: 0.7341 - val_loss: 0.6760
Epoch 21/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 906ms/step - accuracy: 0.6081 - loss: 1.0533
Epoch 21: val_accuracy improved from 0.73408 to 0.74157, saving model to googlenet_googlenet_model.h5


36/36 ━━━━━━━━━━━━━━━━━━━━ 40s 1s/step - accuracy: 0.6079 - loss: 1.0541 - val_accuracy: 0.7416 - val_loss: 0.6571
Epoch 22/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 900ms/step - accuracy: 0.5980 - loss: 1.1067
Epoch 22: val_accuracy improved from 0.74157 to 0.74532, saving model to googlenet_googlenet_model.h5


36/36 ━━━━━━━━━━━━━━━━━━━━ 39s 1s/step - accuracy: 0.5983 - loss: 1.1052 - val_accuracy: 0.7453 - val_loss: 0.6398
Epoch 23/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 905ms/step - accuracy: 0.6067 - loss: 1.0615
Epoch 23: val_accuracy did not improve from 0.74532
36/36 ━━━━━━━━━━━━━━━━━━━━ 39s 1s/step - accuracy: 0.6068 - loss: 1.0611 - val_accuracy: 0.7378 - val_loss: 0.6428
Epoch 24/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 907ms/step - accuracy: 0.6182 - loss: 0.9682
Epoch 24: val_accuracy did not improve from 0.74532
36/36 ━━━━━━━━━━━━━━━━━━━━ 39s 1s/step - accuracy: 0.6183 - loss: 0.9688 - val_accuracy: 0.7378 - val_loss: 0.6422
Epoch 25/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 932ms/step - accuracy: 0.6359 - loss: 0.9635
Epoch 25: val_accuracy did not improve from 0.74532
36/36 ━━━━━━━━━━━━━━━━━━━━ 40s 1s/step - accuracy: 0.6359 - loss: 0.9642 - val_accuracy: 0.7453 - val_loss: 0.6360
Epoch 26/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 904ms/step - accuracy: 0.6358 - loss: 0.9719
Epoch 26: val_accuracy did not impro

36/36 ━━━━━━━━━━━━━━━━━━━━ 40s 1s/step - accuracy: 0.6887 - loss: 0.8825 - val_accuracy: 0.7491 - val_loss: 0.6217
Epoch 29/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 917ms/step - accuracy: 0.6449 - loss: 0.9711
Epoch 29: val_accuracy did not improve from 0.74906
36/36 ━━━━━━━━━━━━━━━━━━━━ 39s 1s/step - accuracy: 0.6444 - loss: 0.9716 - val_accuracy: 0.7491 - val_loss: 0.6145
Epoch 30/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 912ms/step - accuracy: 0.6322 - loss: 0.8880
Epoch 30: val_accuracy improved from 0.74906 to 0.75281, saving model to googlenet_googlenet_model.h5


36/36 ━━━━━━━━━━━━━━━━━━━━ 40s 1s/step - accuracy: 0.6323 - loss: 0.8891 - val_accuracy: 0.7528 - val_loss: 0.6132
Restoring model weights from the end of the best epoch: 30.


# Evaluate on Test Set

In [11]:
test_loss, test_acc = model.evaluate(test_generator)

print("_/ Final Test Accuracy:", test_acc)

7/7 ━━━━━━━━━━━━━━━━━━━━ 5s 681ms/step - accuracy: 0.7364 - loss: 0.8288
_/ Final Test Accuracy: 0.6545454263687134


# Save Training History

In [12]:
with open('googlenet_googlenet_history_inception.pkl', 'wb') as f:
    pickle.dump(history.history, f)

# Prediction on Single Image

In [13]:
import numpy as np
from tensorflow.keras.preprocessing import image

# Load class mapping
with open("class_indices.pkl", "rb") as f:
    class_indices = pickle.load(f)

index_to_class = {v: k for k, v in class_indices.items()}

# Load model
model = tf.keras.models.load_model("googlenet_googlenet_model.h5")

# Load image
img_path = "dataset/test/A2-Sitting-down/235.png"
img = image.load_img(img_path, target_size=(224,224))

img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

# Predict
pred = model.predict(img_array)
pred_class = np.argmax(pred[0])

print("Predicted Class:", index_to_class[pred_class])
print("Confidence:", np.max(pred))

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
Predicted Class: A2-Sitting-down
Confidence: 0.9873655
